# Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data

In [2]:
data = pd.read_csv("data/clean_final_data_mitula_lamudi.csv")
data.head(10)

,Harga_Asli,kecamatan,bedroom,bathrooms,jumlah_guest_toilet,list_facilities,list_buildings,lantai,area_digunakan,luas_tanah,latitude,longitude
0,385000000,Wonoayu,2,1,1,"Internet, Air, Tanpa perabotan, Halaman, Listrik","Keamanan, Taman, Keamanan 24 jam, Area anak-an...",1,34,10,-7.437160,112.622220
1,540000000,Sedati,2,1,1,"Garasi, AC, Kabel video, Air, Tanpa perabotan,...","Keamanan, Gym, Taman, Rumah jaga, Keamanan 24 ...",1,32,32,-7.395451,112.769745
2,1960000000,Candi,4,3,3,"Garasi, AC, Dapur lengkap, Tangki air, Listrik...","Area anak-anak, Rumah jaga, Taman, Keamanan 24...",3,72,32,-7.458483,112.700560
3,950000000,Candi,3,2,2,"Garasi, Halaman, Internet, Tanpa perabotan","Area anak-anak, Taman atap, Taman, Rumah jaga",2,72,32,-7.484486,112.725384
4,691000000,Sukodono,2,2,2,"Garasi, Halaman, Tanpa perabotan, Internet, Li...","Keamanan, Rumah jaga, Taman, Keamanan 24 jam",2,72,32,-7.391429,112.698825
5,650000000,Candi,2,1,1,"Listrik, Fully fenced, Tanpa perabotan",Tidak Ada,1,30,35,-7.484927,112.743517
6,390000000,Sedati,2,1,1,"Garasi, Halaman, Listrik, Fully fenced, Pemand...","Keamanan, Area anak-anak, Rumah jaga, Gym, Tam...",1,35,35,-7.400810,112.803280
7,475000000,Candi,2,1,0,Tanpa perabotan,Tidak Ada,1,35,35,-7.458213,112.690480
8,455000000,Sukodono,2,1,1,Tanpa perabotan,Tidak Ada,1,35,35,-7.388464,112.703773
9,355000000,Tulangan,2,1,1,Tanpa perabotan,Tidak Ada,1,35,35,-7.449429,112.639137


# Hitung jarak lokasi ke destinasi

In [3]:
import pandas as pd
import osmnx as ox
import networkx as nx
import os
import numpy as np
from sklearn.neighbors import BallTree
from tqdm.auto import tqdm

# Aktifkan progress bar pandas
tqdm.pandas()

# ==========================================
# 1. PERSIAPAN DATA & PETA
# ==========================================

# A. Load Data Rumah
# Pastikan nama file sesuai dengan yang ada di screenshot Anda
file_rumah = "data/clean_final_data_mitula_lamudi.csv" 
data_rumah = pd.read_csv(file_rumah)

# B. Load Data Fasilitas (Hasil proses sebelumnya)
# Gunakan file terakhir yang paling lengkap (termasuk bandara & mall)
file_fasilitas = "data/new_fasilitas_sidoarjo_complete_bandara.csv" 
data_fasilitas = pd.read_csv(file_fasilitas)

# C. Load Peta (Graph)
graph_filename = "sidoarjo_drive.graphml"

if os.path.exists(graph_filename):
    print("[INFO] Memuat peta dari penyimpanan lokal (Cepat)...")
    G = ox.load_graphml(graph_filename)
else:
    print("[INFO] Mendownload peta dari OSM (Proses ini butuh waktu)...")
    place_name = "Kabupaten Sidoarjo, Jawa Timur, Indonesia"
    G = ox.graph_from_place(place_name, network_type='drive')
    ox.save_graphml(G, graph_filename)

print("[INFO] Peta siap digunakan.")

# ==========================================
# 2. PRE-PROCESSING NODE (OPTIMASI)
# ==========================================
print("[INFO] Memetakan koordinat rumah ke Node jalan terdekat...")

# Agar cepat, kita cari node terdekat untuk SEMUA rumah sekaligus (vectorized)
# ox.distance.nearest_nodes(G, X, Y) -> X=Longitude, Y=Latitude
house_nodes = ox.distance.nearest_nodes(G, data_rumah['longitude'], data_rumah['latitude'])
data_rumah['origin_node'] = house_nodes

# Kita juga perlu memetakan SEMUA fasilitas ke Node jalan terdekat
print("[INFO] Memetakan koordinat fasilitas ke Node jalan terdekat...")
facility_nodes = ox.distance.nearest_nodes(G, data_fasilitas['longitude'], data_fasilitas['latitude'])
data_fasilitas['dest_node'] = facility_nodes

# ==========================================
# 3. FUNGSI PENGHITUNG JARAK
# ==========================================

def calculate_nearest_distance(df_houses, df_facilities, category_name, graph):
    """
    1. Filter fasilitas berdasarkan kategori.
    2. Cari fasilitas terdekat secara geometris (BallTree).
    3. Hitung jarak jalan raya (NetworkX) ke fasilitas tersebut.
    """
    print(f"\n--- Memproses Kategori: {category_name} ---")
    
    # 1. Filter Kategori
    target_fasilitas = df_facilities[df_facilities['kategori_spesifik'] == category_name].copy()
    
    if target_fasilitas.empty:
        print(f"Warning: Tidak ada data untuk kategori {category_name}")
        return [np.nan] * len(df_houses)

    # 2. Bangun BallTree (Algoritma pencarian tetangga terdekat yang cepat)
    # Konversi ke radian untuk metrik haversine
    fasilitas_rad = np.deg2rad(target_fasilitas[['latitude', 'longitude']].values)
    rumah_rad = np.deg2rad(df_houses[['latitude', 'longitude']].values)
    
    tree = BallTree(fasilitas_rad, metric='haversine')
    
    # Cari indeks fasilitas terdekat untuk setiap rumah (k=1 artinya 1 terdekat)
    # query mengembalikan (jarak, indeks)
    _, indices = tree.query(rumah_rad, k=1)
    
    # Ambil Node ID dari fasilitas yang terpilih
    nearest_dest_nodes = target_fasilitas.iloc[indices.flatten()]['dest_node'].values
    
    # 3. Hitung Jarak Jalan Raya (Looping)
    distances = []
    
    # Kita iterasi menggunakan zip untuk performa lebih baik daripada .apply row-by-row
    # Menggunakan tqdm untuk progress bar
    for orig, dest in tqdm(zip(df_houses['origin_node'], nearest_dest_nodes), total=len(df_houses), desc=f"Hitung Rute {category_name}"):
        try:
            # Hitung jarak terpendek (dalam meter)
            # method='dijkstra' adalah default dan akurat
            dist = nx.shortest_path_length(graph, source=orig, target=dest, weight='length')
            distances.append(dist)
        except nx.NetworkXNoPath:
            # Jika tidak ada jalan (misal beda pulau atau terisolasi)
            distances.append(np.nan)
        except Exception:
            distances.append(np.nan)
            
    return distances

# ==========================================
# 4. EKSEKUSI UTAMA
# ==========================================

# Daftar kategori yang ingin dihitung
list_kategori = [
    'SD', 'SMP', 'SMA/SMK', 'Universitas', 
    'Fasilitas Kesehatan', 
    'Mall', 'Supermarket', 'Minimarket', 
    'Stasiun', 'Terminal', 'Bandara', 'Halte',
    'Entry/Exit Tol'
]

# Loop untuk setiap kategori
for cat in list_kategori:
    # Nama kolom baru, misal: 'dist_SD', 'dist_Mall'
    col_name = f"dist_{cat.replace('/', '_').replace(' ', '_')}"
    
    # Jalankan fungsi
    jarak_meter = calculate_nearest_distance(data_rumah, data_fasilitas, cat, G)
    
    # Simpan ke DataFrame (convert ke KM agar mudah dibaca)
    data_rumah[col_name] = np.array(jarak_meter) / 1000

# ==========================================
# 5. SIMPAN HASIL
# ==========================================

# Hapus kolom temporary 'origin_node' sebelum save
if 'origin_node' in data_rumah.columns:
    data_rumah.drop(columns=['origin_node'], inplace=True)

print("\n[INFO] Selesai. Contoh 5 data teratas:")
# Tampilkan beberapa kolom jarak
cols_show = ['kecamatan'] + [col for col in data_rumah.columns if 'dist_' in col]
display(data_rumah[cols_show].head())

[INFO] Memuat peta dari penyimpanan lokal (Cepat)...
[INFO] Peta siap digunakan.
[INFO] Memetakan koordinat rumah ke Node jalan terdekat...
[INFO] Memetakan koordinat fasilitas ke Node jalan terdekat...

--- Memproses Kategori: SD ---


Hitung Rute SD:   0%|          | 0/1241 [00:00<?, ?it/s]


--- Memproses Kategori: SMP ---


Hitung Rute SMP:   0%|          | 0/1241 [00:00<?, ?it/s]


--- Memproses Kategori: SMA/SMK ---


...

Hitung Rute Halte:   0%|          | 0/1241 [00:00<?, ?it/s]


--- Memproses Kategori: Entry/Exit Tol ---


Hitung Rute Entry/Exit Tol:   0%|          | 0/1241 [00:00<?, ?it/s]


[INFO] Selesai. Contoh 5 data teratas:


,kecamatan,dist_SD,dist_SMP,dist_SMA_SMK,dist_Universitas,dist_Fasilitas_Kesehatan,dist_Mall,dist_Supermarket,dist_Minimarket,dist_Stasiun,dist_Terminal,dist_Bandara,dist_Halte,dist_Entry_Exit_Tol
0,Wonoayu,5.286116,5.405134,5.784943,10.829117,5.527251,9.865963,8.244602,6.644070,5.362372,6.297998,21.386883,5.362372,29.866557
1,Sedati,2.126120,5.450804,1.168335,2.352266,2.209702,6.722774,5.299212,2.808094,4.825077,2.463128,2.463128,4.825077,8.572197
2,Candi,2.267426,2.457838,0.674588,1.755079,1.545930,3.207815,1.426830,1.484812,1.945620,2.891953,14.379988,2.300403,2.036763
3,Candi,1.065887,3.308932,3.562256,4.077568,2.645925,5.483671,3.289849,0.470766,4.540350,3.610592,15.649330,4.453838,14.222444
4,Sukodono,0.544383,2.336779,5.736193,6.735769,2.732387,6.101792,2.181219,1.850357,5.118060,6.280646,9.747785,5.118060,20.407812


In [4]:

output_filename = "data/data_rumah_dengan_jarak_fasilitas.csv"
data_rumah.to_csv(output_filename, index=False)
print(f"\nFile berhasil disimpan ke: {output_filename}")


File berhasil disimpan ke: data/data_rumah_dengan_jarak_fasilitas.csv
